In [4]:
"""
Group 3 - Jumia Egypt: This cleans the raw daily files and loads them into the SQLite database.
"""
import re, sqlite3, sys, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
DB_PATH = BASE_DIR / "jumia_eg.db"

ROOT = BASE_DIR / "data"
RAW_DIR, CLEAN_DIR = ROOT / "raw", ROOT / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_FILE = BASE_DIR / "schema.sql"
REQUIRED = {
    "runs": {"run_id", "source_file", "raw_json_file", "pages_folder", "is_sample", "loaded_at",
             "rows_raw", "rows_clean", "rows_rejected"},
    "categories": {"category_id", "name", "url_path"},
    "brands": {"brand_id", "name"},
    "products": {"product_id", "name", "brand_id", "category_id", "subcategory", "product_url",
                 "first_seen", "last_seen"},
    "observations": {"product_id", "run_id", "scraped_at", "scraped_date", "price_egp", "old_price_egp",
                     "discount_pct", "discount_pct_stated", "is_discounted", "price_is_range", "rating",
                     "review_count", "rank_in_category", "page", "list_name"},
    "rejected_rows": {"run_id", "product_id", "name", "category", "price_text", "reject_reason"},
}

RAW_COLUMNS = ["scraped_at", "category", "category_path", "page", "position_on_page", "product_id", "name", "brand",
               "price_text", "old_price_text", "discount_text", "rating_text", "review_text", "data_price",
               "data_category", "ga4_discount", "ga4_list", "ga4_category", "product_url", "source_page"]


ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩٬٫", "0123456789,.")
NUMBER = re.compile(r"\d[\d,]*\.?\d*")



In [5]:
def norm_text(x):
    """Strip, collapse whitespace, turn missing values into ''."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def numbers_in(text):
    """All numbers in a string: 'EGP 1,234.50' -> [1234.5]; 'EGP 100 - EGP 200' -> [100.0, 200.0]."""
    text = norm_text(text).translate(ARABIC_DIGITS)
    return [float(m.replace(",", "")) for m in NUMBER.findall(text) if m.replace(",", "").replace(".", "")]

def parse_price(text):
    nums = numbers_in(text)
    return (nums[0] if nums else np.nan), (len(nums) > 1)      

def parse_discount(text):
    nums = numbers_in(text)
    return nums[0] if nums else np.nan                         

def parse_rating(text):
    m = re.search(r"(\d(?:\.\d+)?)\s*out of\s*5", norm_text(text).translate(ARABIC_DIGITS))
    return float(m.group(1)) if m else np.nan                   

def parse_reviews(text):
    m = re.search(r"\(\s*([\d,]+)\s*\)", norm_text(text).translate(ARABIC_DIGITS))
    return float(m.group(1).replace(",", "")) if m else np.nan  



In [6]:
def clean(raw):
    """raw: DataFrame read from a raw CSV (all text). Returns (clean_df, rejected_df)."""
    df = raw.copy()
    for col in RAW_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    for col in df.columns:
        if col != "raw_html":
            df[col] = df[col].map(norm_text)
    n_raw = len(df)

    # Timestamps
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")
    df["scraped_date"] = df["scraped_at"].dt.strftime("%Y-%m-%d")

    # Prices (strip 'EGP', commas, symbols)
    parsed = df["price_text"].map(parse_price)
    df["price_egp"] = parsed.map(lambda t: t[0])
    df["price_is_range"] = parsed.map(lambda t: t[1])
    fallback = df["data_price"].map(lambda t: parse_price(t)[0])
    df["price_egp"] = df["price_egp"].fillna(fallback)
    df["old_price_egp"] = df["old_price_text"].map(lambda t: parse_price(t)[0])

    # Discount: keep what the site states, and compute the real one from old vs new price 
    df["discount_pct_stated"] = df["discount_text"].map(parse_discount)
    bad_old = df["old_price_egp"].notna() & (df["old_price_egp"] <= df["price_egp"])
    df.loc[bad_old, "old_price_egp"] = np.nan                   
    calc = (df["old_price_egp"] - df["price_egp"]) / df["old_price_egp"] * 100
    df["discount_pct"] = calc.round(1)
    df.loc[df["discount_pct"].isna(), "discount_pct"] = df["discount_pct_stated"]   
    df.loc[~df["discount_pct"].between(0, 100), "discount_pct"] = np.nan
    df["is_discounted"] = df["discount_pct"].fillna(0) > 0

    # Rating "4.5 out of 5" -> 4.5, reviews "(123)" -> 123. Missing stays missing (NaN), it is NOT zero.
    df["rating"] = df["rating_text"].map(parse_rating)
    df.loc[~df["rating"].between(0, 5), "rating"] = np.nan
    df["review_count"] = df["review_text"].map(parse_reviews)

    # Text standardisation
    df["name"] = df["name"].str.replace(r"\s+", " ", regex=True)
    key = df["brand"].str.casefold()
    canonical = (df.loc[key != "", ["brand"]].assign(k=key[key != ""]).groupby("k")["brand"]
                 .agg(lambda s: s.value_counts().idxmax()))      # most common spelling of each brand
    df["brand"] = key.map(canonical).fillna("Unknown")
    parts = df["ga4_category"].str.split(">").map(lambda p: [x.strip() for x in p if x.strip()])
    df["subcategory"] = parts.map(lambda p: p[1] if len(p) > 1 else "")
    df["product_id"] = df["product_id"].where(df["product_id"] != "", df["product_url"])   # fall back to URL

    # Rank inside the category = order shown on the site (page, then position on page)
    df["page"] = pd.to_numeric(df["page"], errors="coerce")
    df["position_on_page"] = pd.to_numeric(df["position_on_page"], errors="coerce")
    df = df.sort_values(["category", "page", "position_on_page"], kind="stable")

    # Rejections (kept in a file, with the reason)
    reasons = pd.Series("", index=df.index)
    reasons[df["product_id"] == ""] = "no product id or url"
    reasons[(reasons == "") & df["scraped_at"].isna()] = "bad timestamp"
    reasons[(reasons == "") & (df["name"] == "")] = "no product name"
    reasons[(reasons == "") & (df["price_egp"].isna() | (df["price_egp"] <= 0))] = "missing or invalid price"
    dup = df.duplicated(["product_id", "scraped_date"], keep="first")
    reasons[(reasons == "") & dup] = "duplicate product on the same day"
    rejected = df[reasons != ""].assign(reject_reason=reasons[reasons != ""])
    good = df[reasons == ""].copy()
    good["rank_in_category"] = good.groupby("category").cumcount() + 1   # order shown on the site, after removals
    good["review_count"] = good["review_count"].astype("Int64")
    print(f"  rows: {n_raw} raw -> {len(good)} clean, {len(rejected)} rejected")
    if len(rejected):
        print("  reasons:", rejected["reject_reason"].value_counts().to_dict())
    return good, rejected



In [7]:
def ensure_schema(con):
    if not SCHEMA_FILE.exists():
        sys.exit(f"schema.sql not found next to the script ({SCHEMA_FILE}).")
    for table, needed in REQUIRED.items():
        existing = {r[1] for r in con.execute(f"PRAGMA table_info({table})")}
        if existing and needed - existing:
            sys.exit(f"\nThe table '{table}' already exists in {DB_PATH.name} but lacks columns: {sorted(needed - existing)}.\n"
                     "This database uses an older layout. Use a new database file (change DB_PATH) so schema.sql can create the tables,\n"
                     "or send the output of  sqlite3 <db> .schema  and the script will be adapted.")
    con.executescript(SCHEMA_FILE.read_text(encoding="utf-8"))    # CREATE ... IF NOT EXISTS, safe to repeat
    con.commit()





In [8]:
def none_if_nan(v):
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA:
        return None
    return v.item() if hasattr(v, "item") else v

In [9]:
def lookup_id(cur, table, id_col, name, extra_col=None, extra_val=None):
    """Insert a category/brand once and return its id."""
    if extra_col:
        cur.execute(f"INSERT INTO {table} (name, {extra_col}) VALUES (?, ?) "
                    f"ON CONFLICT(name) DO UPDATE SET {extra_col} = excluded.{extra_col}", (name, extra_val))
    else:
        cur.execute(f"INSERT OR IGNORE INTO {table} (name) VALUES (?)", (name,))
    return cur.execute(f"SELECT {id_col} FROM {table} WHERE name = ?", (name,)).fetchone()[0]



In [10]:
def load(con, good, rejected, csv_path, n_raw):
    cur = con.cursor()
    pages = ""
    if (good["source_page"] != "").any():
        pages = Path(good.loc[good["source_page"] != "", "source_page"].iloc[0]).parent.as_posix()
    cur.execute("""INSERT INTO runs (source_file, raw_json_file, pages_folder, is_sample, loaded_at,
                                     rows_raw, rows_clean, rows_rejected)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                   ON CONFLICT(source_file) DO UPDATE SET raw_json_file = excluded.raw_json_file,
                       pages_folder = excluded.pages_folder, is_sample = excluded.is_sample,
                       loaded_at = excluded.loaded_at, rows_raw = excluded.rows_raw,
                       rows_clean = excluded.rows_clean, rows_rejected = excluded.rows_rejected""",
                (csv_path.name, csv_path.with_suffix(".json").name, pages, int(csv_path.name.startswith("sample_")),
                 dt.datetime.now().isoformat(timespec="seconds"), n_raw, len(good), len(rejected)))
    run_id = cur.execute("SELECT run_id FROM runs WHERE source_file = ?", (csv_path.name,)).fetchone()[0]

    cat_ids = {r.category: lookup_id(cur, "categories", "category_id", r.category, "url_path", r.category_path)
               for r in good.drop_duplicates("category").itertuples()}
    brand_ids = {b: lookup_id(cur, "brands", "brand_id", b) for b in good["brand"].unique()}

    products = [(r.product_id, r.name, brand_ids[r.brand], cat_ids[r.category], r.subcategory, r.product_url,
                 r.scraped_date, r.scraped_date) for r in good.itertuples()]
    cur.executemany("""INSERT INTO products (product_id, name, brand_id, category_id, subcategory, product_url,
                                             first_seen, last_seen)
                       VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                       ON CONFLICT(product_id) DO UPDATE SET name = excluded.name, brand_id = excluded.brand_id,
                           category_id = excluded.category_id, subcategory = excluded.subcategory,
                           product_url = excluded.product_url,
                           first_seen = MIN(first_seen, excluded.first_seen),
                           last_seen = MAX(last_seen, excluded.last_seen)""", products)

    obs = [(r.product_id, run_id, r.scraped_at.isoformat(timespec="seconds"), r.scraped_date,
            none_if_nan(r.price_egp), none_if_nan(r.old_price_egp), none_if_nan(r.discount_pct),
            none_if_nan(r.discount_pct_stated), int(r.is_discounted), int(r.price_is_range),
            none_if_nan(r.rating), none_if_nan(r.review_count), int(r.rank_in_category),
            none_if_nan(r.page), r.ga4_list or None) for r in good.itertuples()]
    cur.executemany("""INSERT INTO observations (product_id, run_id, scraped_at, scraped_date, price_egp, old_price_egp,
                           discount_pct, discount_pct_stated, is_discounted, price_is_range, rating, review_count,
                           rank_in_category, page, list_name)
                       VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                       ON CONFLICT(product_id, scraped_date) DO UPDATE SET run_id = excluded.run_id,
                           scraped_at = excluded.scraped_at, price_egp = excluded.price_egp,
                           old_price_egp = excluded.old_price_egp, discount_pct = excluded.discount_pct,
                           discount_pct_stated = excluded.discount_pct_stated, is_discounted = excluded.is_discounted,
                           price_is_range = excluded.price_is_range, rating = excluded.rating,
                           review_count = excluded.review_count, rank_in_category = excluded.rank_in_category,
                           page = excluded.page, list_name = excluded.list_name""", obs)

    cur.execute("DELETE FROM rejected_rows WHERE run_id = ?", (run_id,))     # reload-safe
    cur.executemany("""INSERT INTO rejected_rows (run_id, product_id, name, category, price_text, reject_reason)
                       VALUES (?, ?, ?, ?, ?, ?)""",
                    [(run_id, r.product_id or None, r.name or None, r.category or None, r.price_text or None,
                      r.reject_reason) for r in rejected.itertuples()])
    con.commit()

In [11]:
def main():
    args = [a for a in sys.argv[1:] if not a.startswith("--")]
    if args:
        files = [Path(a) for a in args]
    else:
        files = sorted(RAW_DIR.glob("jumia_eg_*.csv"))
        if "--include-sample" in sys.argv:
            files += sorted(RAW_DIR.glob("sample_jumia_eg_*.csv"))
    if not files:
        sys.exit(f"No raw CSV files found in {RAW_DIR}")

    con = sqlite3.connect(DB_PATH)
    con.execute("PRAGMA foreign_keys = ON")
    ensure_schema(con)
    for f in files:
        print(f"\n{f.name}")
        raw = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8-sig")
        good, rejected = clean(raw)
        good.drop(columns=["raw_html"], errors="ignore").to_csv(CLEAN_DIR / f"{f.stem}_clean.csv", index=False, encoding="utf-8-sig")
        rejected.drop(columns=["raw_html"], errors="ignore").to_csv(CLEAN_DIR / f"{f.stem}_rejected.csv", index=False, encoding="utf-8-sig")
        load(con, good, rejected, f, len(raw))
    print("\nDatabase totals:")
    for t in ("runs", "categories", "brands", "products", "observations", "rejected_rows"):
        print(f"  {t:13s} {con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]} rows")
    print("Database file:", DB_PATH.resolve())
    con.close()

if __name__ == "__main__":
    main()



jumia_eg_2026-10-13_1909.csv
  rows: 714 raw -> 714 clean, 0 rejected

Database totals:
  runs          1 rows
  categories    3 rows
  brands        155 rows
  products      714 rows
  observations  714 rows
  rejected_rows 0 rows
Database file: /home/sam/Downloads/Data-mining-Assignment-Group-3/jumia_eg.db
